[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/JAXCode/blob/master/solutions/b_24_embedding_pure_solution.ipynb)

# 🟢 Solution: Embedding without Flax

*Core Ops & Layers · Easy*

Reference implementation. Try it yourself in `b_24_embedding_pure.ipynb` first.

---
Problem 18's embedding table, with a plain pytree instead of an `nnx.Module`.

### Signature
```python
def init_embedding(key, num_embeddings, embedding_dim):
    ...   # -> {"table": (num_embeddings, embedding_dim)}

def apply_embedding(params, indices):
    ...   # (...) int -> (..., embedding_dim)

def attend_embedding(params, x):
    ...   # (..., embedding_dim) -> (..., num_embeddings)
```

Initialise the table with `jax.random.normal(...) * 0.02` — the GPT-2
convention, same as problem 18.

### What changes, and what does not
The **maths is identical**. What goes away is the wrapper:

```python
self.table[indices]          # 18: an nnx.Param, which proxies to the array
params["table"][indices]     # here: it IS the array
x @ self.table[...].T        # 18: [...] to unwrap explicitly
x @ params["table"].T        # here: nothing to unwrap
```

Every question about `.value` vs `[...]` vs `.get_value()` simply stops
existing. That is the trade: you lose the module's bookkeeping and you gain
one less layer between you and the array.

### Weight tying is now obvious
`attend_embedding` reuses the same array as `apply_embedding`, which is the
whole point of weight tying — and with an explicit pytree you can *see* that
there is only one `table` in it, rather than trusting a module to share it.

In [ ]:
# Colab setup (no-op when running locally).
# jax-judge is not published on PyPI, so the judge is installed from the
# repo itself. Regenerate with JAXCODE_REPO=you/YourFork to point this at
# your own fork:  JAXCODE_REPO=you/JAXCode make notebooks
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q flax optax')
    get_ipython().run_line_magic(
        'pip', 'install -q git+https://github.com/YOUR-GITHUB-USERNAME/JAXCode.git')
except ImportError:
    pass

In [ ]:
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "|", jax.devices())

In [ ]:
# ✅ REFERENCE SOLUTION

import jax
import jax.numpy as jnp


def init_embedding(key, num_embeddings, embedding_dim):
    return {
        "table": jax.random.normal(key, (num_embeddings, embedding_dim)) * 0.02
    }


def apply_embedding(params, indices):
    # A gather. Advanced indexing already handles any leading shape, and it is
    # O(1) per token instead of the O(V) a one-hot matmul would cost.
    return params["table"][indices]


def attend_embedding(params, x):
    # Weight tying: the same array, transposed.
    return x @ params["table"].T

In [ ]:
# 🔍 Verify
import jax
import jax.numpy as jnp

params = init_embedding(jax.random.key(0), 100, 8)
print("table:", params["table"].shape)

for idx in [jnp.array(5), jnp.array([1, 2, 3]), jnp.zeros((2, 4), dtype=jnp.int32)]:
    print(f"  indices {str(idx.shape):<8} -> {apply_embedding(params, idx).shape}")

print("attend:", attend_embedding(params, jnp.ones((2, 4, 8))).shape)

# Repeated indices must accumulate in the gradient.
g = jax.grad(lambda p: jnp.sum(apply_embedding(p, jnp.array([1, 1, 2]))))(params)
print("\ngrad row 1 (used twice):", g["table"][1, 0])
print("grad row 2 (used once): ", g["table"][2, 0])

In [ ]:
# Run the judge against the reference solution
from jax_judge import check

check("embedding_pure")